In [2]:
!pip install -q \
  "langchain" \
  "langchain-community" \
  "langchain-core" \
  "langchain-text-splitters" \
  "langchain-huggingface" \
  "sentence-transformers"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
pip install -U langchain langchain-community langchain-experimental langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 6.6 MB/s eta 0:00:00


In [4]:
!pip uninstall -y torchcodec sentence-transformers
!pip install -q \
  "sentence-transformers==2.7.0" \
  "langchain==0.1.20" \
  "langchain-community==0.0.38" \
  "langchain-experimental==0.0.57" \
  "langchain-text-splitters==0.0.2"

Found existing installation: torchcodec 0.10.0
Uninstalling torchcodec-0.10.0:
  Successfully uninstalled torchcodec-0.10.0
Found existing installation: sentence-transformers 5.4.1
Uninstalling sentence-transformers-5.4.1:
  Successfully uninstalled sentence-transformers-5.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.4/193.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 68.3 MB/s eta 0:00:00
   ━━

In [5]:
# Libraries
# Fix-chunking
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# Load dataset
df = pd.read_csv("/content/sample_data/customer_support_tickets.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
documents = df["ticket_description"].dropna().tolist()

In [ ]:
# Naive chunking
naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

In [ ]:
# Semantic chunking
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
semantic_splitter = SemanticChunker(
    embeddings=embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85  # split when content shifts by > 85th percentile
)

In [ ]:
!pip uninstall -y simsimd

In [ ]:
# Compare on first 100 tickets
naive_chunks = []
semantic_chunks = []

for doc in documents[:100]:
    naive_chunks.extend(naive_splitter.split_text(doc))
    semantic_chunks.extend(semantic_splitter.split_text(doc))

print("=" * 50)
print("CHUNKING COMPARISON (first 100 tickets)")
print("=" * 50)
print(f"Naive chunks   : {len(naive_chunks):>5} | avg length: {sum(len(c) for c in naive_chunks)/len(naive_chunks):.0f} chars")
print(f"Semantic chunks: {len(semantic_chunks):>5} | avg length: {sum(len(c) for c in semantic_chunks)/len(semantic_chunks):.0f} chars")

In [ ]:
# Show a side-by-side example
sample = documents[0]
print("\n── Sample document ──")
print(sample[:400])

print("\n── Naive split ──")
for i, c in enumerate(naive_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {c[:120]}...")

print("\n── Semantic split ──")
for i, c in enumerate(semantic_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {c[:120]}...")

# Fix #2 : Embedding Model Comparison #
*What this shows*: A generic embedding model treats domain-specific synonyms as different. A domain-adapted model scores them as similar, improving retrieval relevance.

In [ ]:
# fix_2_embeddings.py
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Load two embedding models
generic_model = SentenceTransformer("all-MiniLM-L6-v2")
domain_model  = SentenceTransformer("BAAI/bge-base-en-v1.5")

In [ ]:

# Domain-specific query and semantically equivalent documents
test_cases = [
    {
        "query": "My subscription was charged twice this month",
        "relevant": "Duplicate billing issue — customer charged twice in billing cycle",
        "irrelevant": "How do I update my payment method in account settings?"
    },
    {
        "query": "The API gateway returned a 429 throttle error on Pro tier",
        "relevant": "User is being rate limited on their current plan",
        "irrelevant": "Upgrade your plan to get more API calls per minute"
    },
    {
        "query": "I cannot log into my account after the password reset",
        "relevant": "Authentication failure after password change — session token invalid",
        "irrelevant": "Our servers experienced downtime on March 12th"
    }
]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=" * 65)
print(f"{'Test Case':<5} | {'Model':<10} | {'Relevant Sim':>12} | {'Irrelevant Sim':>14}")
print("=" * 65)

for i, case in enumerate(test_cases):
    for model_name, model in [("Generic", generic_model), ("Domain", domain_model)]:
        q   = model.encode(case["query"])
        rel = model.encode(case["relevant"])
        irr = model.encode(case["irrelevant"])

        sim_rel = cosine_sim(q, rel)
        sim_irr = cosine_sim(q, irr)

        flag = "✅" if sim_rel > sim_irr else "❌"
        print(f"  TC {i+1}   | {model_name:<10} | {sim_rel:>12.4f} | {sim_irr:>14.4f}  {flag}")
    print("-" * 65)

# Fix #3 : Cross-Encoder Reranking
*What this shows*: Embedding similarity retrieves by topic overlap. A cross-encoder reranks by actual query-document relevance, filtering noise before it reaches the LLM.


In [ ]:
# fix_3_reranking.py
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
reranker        = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
# Simulate a query and 10 candidates from vector DB
query = "How do I get a refund for a duplicate charge?"

candidate_chunks = [
    "To request a refund, contact billing support with your order ID.",           # relevant
    "Refunds for duplicate charges are processed within 3-5 business days.",      # highly relevant
    "Our return policy covers physical items within 30 days of purchase.",        # off-topic
    "If you see an unexpected charge, review your subscription plan first.",      # partially relevant
    "Double billing issues are escalated to the finance team automatically.",     # relevant
    "You can update your payment method from the account settings page.",         # off-topic
    "Subscription upgrades take effect at the start of the next billing cycle.",  # off-topic
    "Contact support if your refund has not arrived after 7 business days.",      # relevant
    "Billing disputes can be submitted through the Help Center portal.",          # relevant
    "Account charges are visible under the Billing tab in your dashboard.",       # off-topic
]

In [ ]:
# Step 1: Embedding similarity ranking (naive approach)
q_emb = embedding_model.encode(query)
emb_scores = [
    np.dot(q_emb, embedding_model.encode(chunk)) /
    (np.linalg.norm(q_emb) * np.linalg.norm(embedding_model.encode(chunk)))
    for chunk in candidate_chunks
]
emb_ranked = sorted(zip(emb_scores, candidate_chunks), reverse=True)


In [ ]:
# Step 2: Cross-encoder reranking
pairs  = [(query, chunk) for chunk in candidate_chunks]
scores = reranker.predict(pairs)
reranked = sorted(zip(scores, candidate_chunks), reverse=True)


In [ ]:
# Compare
print("TOP 5 — Embedding Similarity (before reranking):")
print("-" * 65)
for score, chunk in emb_ranked[:5]:
    print(f"  [{score:.4f}] {chunk}")

print("\nTOP 5 — Cross-Encoder Reranking (after reranking):")
print("-" * 65)
for score, chunk in reranked[:5]:
    print(f"  [{score:.3f}] {chunk}")

# Fix #4 — Context Compression
*What this shows*: Passing 10 full chunks to the LLM introduces noise and triggers the "Lost in the Middle" failure. Compressing each chunk to only the sentences relevant to the query keeps the prompt short and the signal high.


In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys

# uninstall everything related
!{sys.executable} -m pip uninstall -y langchain langchain-core langchain-community langchain-experimental langchain-text-splitters

# reinstall clean versions
!{sys.executable} -m pip install -U langchain langchain-community langchain-experimental langchain-text-splitters faiss-cpu sentence-transformers

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "--force-reinstall", "pandas"])

In [ ]:
!pip install langchain==0.1.20 langchain-community langchain-experimental langchain-text-splitters

In [ ]:
!pip install -q --upgrade pandas

In [ ]:
!pip install -q \
  "numpy<2.0" \
  "pandas==2.2.2" \
  "langchain==0.1.20" \
  "langchain-community==0.0.38" \
  "langchain-core==0.1.53" \
  "langchain-text-splitters==0.0.2" \
  "sentence-transformers" \
  "scikit-learn" \
  "scipy"

In [ ]:
#  Load and chunk the dataset

df = pd.read_csv("/content/sample_data/customer_support_tickets.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
documents = df["ticket_description"].dropna().tolist()
documents = df["ticket_description"].dropna().tolist()
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
splitter   = SemanticChunker(embeddings=embeddings, breakpoint_threshold_type="percentile")

chunks = []
for doc in documents[:300]:   # use first 300 tickets for demo
    chunks.extend(splitter.split_text(doc))

print(f"Total semantic chunks indexed: {len(chunks)}")

In [ ]:
!pip install faiss-gpu

In [ ]:
# Build vector store
vectorstore    = FAISS.from_texts(chunks, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
